# DoCIT · iTrust trace-link recovery baselines (V2.1)
**Research direction:** *Verified Domain Knowledge for Requirements-to-Code Traceability*.  
**This notebook covers Task 1 only:** reproducible, original-requirement retrieval baselines, **not** evidence that a domain-knowledge layer has improved trace recovery.

**Data:** `finegrained-traceability-main(1).zip` (iTrust subset) · 131 requirements · 226 Java candidates · 286 recorded links involving 105 requirements.

**Workflow:** load/check original data → TF-IDF → optional UniXcoder ablations → evaluate against reference links → failure analysis → export results. All models rank **the same 226 Java files**.

### How to run in Google Colab
1. Upload this `.ipynb` to Colab. Set **Runtime → Change runtime type → T4 GPU** if running UniXcoder.
2. In the configuration cell below, use `ZIP_PATH` to point to your Google Drive ZIP, or leave it empty to upload the ZIP when prompted. If your ZIP is in Drive, mount Drive first.
3. Choose **Runtime → Run all**. The TF-IDF baseline always runs; UniXcoder runs automatically when GPU is available, or when explicitly enabled.
4. Download the results ZIP created at the end, and save the executed notebook to Drive.

**Ethics of evaluation:** The solution matrix is read solely to score the generated ranks. It must **not** be used to expand text, author a glossary, or tune on a reserved test partition. The 105-query results in this baseline notebook are exploratory and are not a held-out test of the future knowledge-based method.

## 0. Configuration
- `RUN_UNIXCODER = "auto"` runs UniXcoder only if a CUDA GPU is available. Set to `True` to force CPU inference (slow), or `False` for TF-IDF only.
- If the ZIP is already in Google Drive: set `ZIP_PATH = "/content/drive/MyDrive/.../finegrained-traceability-main(1).zip"`; set `MOUNT_DRIVE=True`.
- For a fresh Colab, leaving `ZIP_PATH=""` will prompt for the archive. We extract **only** the iTrust dataset, not the whole repository.

In [11]:
# Configuration: edit here, not throughout the notebook.
ZIP_PATH = ""                   # ZIP path, or leave blank for Colab upload
MOUNT_DRIVE = False              # True if ZIP_PATH is in Google Drive
RUN_UNIXCODER = "auto"          # "auto" / True / False
MODEL_NAME = "microsoft/unixcoder-base"
MODEL_REVISION = "main"         # Record the resolved revision in outputs
K = 5
SEED = 42
PILOT_REQ = "UC27S1.txt"
MAX_TOKENS = 512
CHUNK_STRIDE = 384
BATCH_SIZE = 8                   # increase to 16/32 if GPU memory permits
USE_AMP = False                 # float32 default, for stability
WORK_DIR = "/content/docit_task1"
# In a local Python runtime (not Colab), use /mnt/data/docit_task1 by default.
import os
if not os.path.exists("/content") and WORK_DIR == "/content/docit_task1":
    WORK_DIR = "/mnt/data/docit_task1"
OUTPUT_DIR = os.path.join(WORK_DIR, "results")
CACHE_DIR = os.path.join(WORK_DIR, "embedding_cache")

## 1. Load and verify iTrust
The ZIP uses a nested folder (`finegrained-traceability-main/datasets/iTrust/`). The loader finds that subtree, retains exact filenames, and fails loudly on missing or duplicate IDs. It also records the archive hash for reproducibility.

In [2]:
import hashlib, json, platform, re, zipfile, sys, time
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import scipy, sklearn

np.random.seed(SEED)
work = Path(WORK_DIR)
output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)
Path(CACHE_DIR).mkdir(parents=True, exist_ok=True)

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

if ZIP_PATH:
    zip_path = Path(ZIP_PATH).expanduser()
else:
    candidates = [*Path("/content").glob("*finegrained*traceability*.zip")] if Path("/content").exists() else []
    candidates += list(Path("/mnt/data").glob("*finegrained*traceability*.zip")) if Path("/mnt/data").exists() else []
    if candidates:
        zip_path = sorted(candidates)[0]
        print("Auto-detected ZIP:", zip_path)
    else:
        try:
            from google.colab import files
        except ImportError as exc:
            raise FileNotFoundError("Provide ZIP_PATH to the finegrained-traceability ZIP.") from exc
        print("Upload your finegrained-traceability ZIP now.")
        uploaded = files.upload()
        names = [x for x in uploaded if x.lower().endswith('.zip')]
        if len(names) != 1:
            raise ValueError("Upload exactly one ZIP archive.")
        zip_path = Path(names[0])

if not zip_path.is_file():
    raise FileNotFoundError(f"ZIP not found: {zip_path}")
archive_sha256 = hashlib.sha256(zip_path.read_bytes()).hexdigest()
print("Archive:", zip_path.name, "SHA-256:", archive_sha256)

# Read directly from ZIP; no extraction, path traversal, or Google Drive copying required.
with zipfile.ZipFile(zip_path) as zf:
    names = [x for x in zf.namelist() if "/datasets/iTrust/" in x and not x.endswith("/")]
    prefixes = {x.split("/datasets/iTrust/", 1)[0] + "/datasets/iTrust/" for x in names}
    if len(prefixes) != 1:
        raise ValueError(f"Expected one iTrust dataset root, found {prefixes}")
    prefix = next(iter(prefixes))

    def load_group(group, ext):
        items = {}
        for name in sorted(names):
            relative = name[len(prefix):]
            if relative.startswith(group + "/") and relative.endswith(ext):
                leaf = Path(relative).name
                if leaf in items:
                    raise ValueError(f"Ambiguous duplicate {group} filename: {leaf}")
                items[leaf] = zf.read(name).decode("utf-8", errors="replace")
        return items

    requirements = load_group("req", ".txt")
    source_code = load_group("code", ".java")
    gt_file = prefix + "itrust_solution_links.txt"
    if gt_file not in zf.namelist():
        raise FileNotFoundError("Missing iTrust solution matrix in archive")
    solution_text = zf.read(gt_file).decode("utf-8")

ground_truth = defaultdict(set)
for lineno, line in enumerate(solution_text.splitlines(), start=1):
    if not line.strip():
        continue
    if line.count(":") != 1:
        raise ValueError(f"Malformed solution-matrix entry on line {lineno}: {line!r}")
    req_id, code_id = [x.strip() for x in line.split(":", 1)]
    ground_truth[req_id].add(code_id)
ground_truth = dict(ground_truth)
missing_req = sorted(set(ground_truth) - set(requirements))
missing_code = sorted({c for links in ground_truth.values() for c in links} - set(source_code))
assert not missing_req and not missing_code, (missing_req, missing_code)
assert len(requirements) == 131 and len(source_code) == 226, "This is not the expected iTrust version. Recheck the archive."
assert len(ground_truth) == 105 and sum(map(len, ground_truth.values())) == 286, "Unexpected solution matrix; investigate before scoring."
print(f"Loaded {len(requirements)} requirement files, {len(source_code)} Java files, {len(ground_truth)} linked requirements, {sum(map(len, ground_truth.values()))} gold links.")
print(f"Unlinked requirement IDs (unknown status, not confirmed negatives): {len(set(requirements)-set(ground_truth))}")

Upload your finegrained-traceability ZIP now.


Saving finegrained-traceability-main.zip to finegrained-traceability-main.zip
Archive: finegrained-traceability-main.zip SHA-256: b7df2936d6304ccbe75c20f7f96f431b23861f0f9bbb90325774efade39bda62
Loaded 131 requirement files, 226 Java files, 105 linked requirements, 286 gold links.
Unlinked requirement IDs (unknown status, not confirmed negatives): 26


## 2. Identical candidate sets, evaluation, and historical pilot
All similarity matrices have rows in `req_ids` order and columns in `code_ids` order. We report **macro averages across the 105 linked queries**, never treating 26 unrecorded cases as known negatives. `AP` uses the entire 226-item ranking, not only the top five.

In [3]:
req_ids = sorted(requirements)
code_ids = sorted(source_code)
req_pos = {r: i for i, r in enumerate(req_ids)}
code_pos = {c: j for j, c in enumerate(code_ids)}
eval_reqs = [r for r in req_ids if r in ground_truth]
assert PILOT_REQ in ground_truth
rng = np.random.default_rng(SEED)
others = sorted(set(ground_truth) - {PILOT_REQ})
pilot_reqs = [PILOT_REQ] + list(rng.choice(others, size=9, replace=False))


def evaluate(similarity, ids=eval_reqs, k=K):
    similarity = np.asarray(similarity)
    if similarity.shape != (len(req_ids), len(code_ids)):
        raise ValueError(f"Wrong score shape {similarity.shape}")
    if not np.isfinite(similarity).all():
        raise ValueError("Non-finite similarity scores")
    rows = []
    for req_id in ids:
        ordered = np.argsort(-similarity[req_pos[req_id]], kind="stable")
        ranks = np.empty(len(code_ids), dtype=int)
        ranks[ordered] = np.arange(1, len(code_ids) + 1)
        true_ranks = sorted(ranks[code_pos[c]] for c in ground_truth[req_id])
        hits = sum(r <= k for r in true_ranks)
        p = hits / k
        r = hits / len(ground_truth[req_id])
        f1 = 2 * p * r / (p + r) if p + r else 0.0
        ap = sum(i / rank for i, rank in enumerate(true_ranks, start=1)) / len(true_ranks)
        rows.append({"Requirement": req_id, f"Precision@{k}": p, f"Recall@{k}": r,
                     f"F1@{k}": f1, "AP": ap, "FirstHitRank": true_ranks[0],
                     "GoldLinkCount": len(true_ranks)})
    return pd.DataFrame(rows).set_index("Requirement")


def summarize(table):
    out = table[[f"Precision@{K}", f"Recall@{K}", f"F1@{K}", "AP", "FirstHitRank"]].mean()
    return out.rename({"AP": "MAP", "FirstHitRank": "MeanFirstHitRank"})


def top_matches(scores, req_id, n=10):
    order = np.argsort(-scores[req_pos[req_id]], kind="stable")[:n]
    return pd.DataFrame({"Rank": list(range(1, len(order)+1)),
                         "JavaFile": [code_ids[j] for j in order],
                         "Similarity": scores[req_pos[req_id], order],
                         "RecordedLink": [code_ids[j] in ground_truth.get(req_id, set()) for j in order]})

print("Linked query count:", len(eval_reqs))
print("10-query historical sample:", pilot_reqs)
print("Worked example:", requirements[PILOT_REQ][:1500])
print("Recorded links:", sorted(ground_truth[PILOT_REQ]))

# Evaluator unit test only (NOT an experiment): all gold links placed above non-links -> AP=1.
perfect = np.zeros((len(req_ids), len(code_ids)), dtype=float)
for rid, matches in ground_truth.items():
    for code_id in matches:
        perfect[req_pos[rid], code_pos[code_id]] = 1.0
assert np.allclose(evaluate(perfect)["AP"].to_numpy(), 1.0)
print("Evaluator sanity check passed: perfect ranking gives MAP=1.")

Linked query count: 105
10-query historical sample: ['UC27S1.txt', np.str_('UC18E1.txt'), np.str_('UC36S1.txt'), np.str_('UC3S2.txt'), np.str_('UC34E2.txt'), np.str_('UC26S2.txt'), np.str_('UC34S5.txt'), np.str_('UC8S2.txt'), np.str_('UC6S3.txt'), np.str_('UC12.txt')]
Worked example: The user has successfully changed his/her password (UC3, S2). An email informing the user of the password change is sent to the user including the MID but not the password.

Recorded links: ['EmailUtil.java', 'ResetPasswordAction.java']
Evaluator sanity check passed: perfect ranking gives MAP=1.


## 3. Reproduce V2 TF-IDF baseline
For historical comparability, the vectorizer fits on the original **131 requirement documents + 226 code documents** (a transductive, unsupervised representation). The target file name is added to its text. No solution-matrix links enter the representation or ranking.

In [4]:
def preprocess_text(text):
    text = re.sub(r"([a-z0-9])([A-Z])", r"\1 \2", text)
    text = re.sub(r"([A-Z]+)([A-Z][a-z])", r"\1 \2", text)
    return text.replace("_", " ").lower()

req_raw = [requirements[r] for r in req_ids]
code_raw = [c + "\n" + source_code[c] for c in code_ids]
req_clean = [preprocess_text(s) for s in req_raw]
code_clean = [preprocess_text(s) for s in code_raw]

vectorizer = TfidfVectorizer(stop_words="english", ngram_range=(1,2), sublinear_tf=True)
vectors = vectorizer.fit_transform(req_clean + code_clean)
tfidf_similarity = cosine_similarity(vectors[:len(req_ids)], vectors[len(req_ids):])
all_similarities = {"TF-IDF": tfidf_similarity}
all_results = {"TF-IDF": evaluate(tfidf_similarity)}
print("Vocabulary:", len(vectorizer.vocabulary_), "; score matrix:", tfidf_similarity.shape)
display(pd.DataFrame({"TF-IDF": summarize(all_results["TF-IDF"])}).T.round(4))
print("Historical 10-query pilot (same input settings):")
display(pd.DataFrame({"TF-IDF": summarize(all_results["TF-IDF"].loc[pilot_reqs])}).T.round(4))
display(top_matches(tfidf_similarity, PILOT_REQ, 10))

Vocabulary: 18923 ; score matrix: (131, 226)


,Precision@5,Recall@5,F1@5,MAP,MeanFirstHitRank
TF-IDF,0.1924,0.3893,0.2437,0.2896,14.3429


Historical 10-query pilot (same input settings):


,Precision@5,Recall@5,F1@5,MAP,MeanFirstHitRank
TF-IDF,0.24,0.6,0.3178,0.3765,2.2


,Rank,JavaFile,Similarity,RecordedLink
0,1,ResetPasswordAction.java,0.086355,True
1,2,AddPatientAction.java,0.053977,False
2,3,AddUAPAction.java,0.051473,False
3,4,AddHCPAction.java,0.050585,False
4,5,AddPHAAction.java,0.050096,False
5,6,AddERespAction.java,0.050084,False
6,7,RandomPassword.java,0.045724,False
7,8,AuthDAO.java,0.043787,False
8,9,PersonnelBean.java,0.039801,False
9,10,LabProcUAPAction.java,0.033503,False


## 4. UniXcoder encoder-only model (optional GPU run)
The original V2 added encoder-only tokens, overlapping windows, and three ablation variants. This version uses the checkpoint's `is_decoder=True` configuration and an explicit **bidirectional attention mask** as in Microsoft's official UniXcoder implementation. The vector for a chunk is the mask-weighted mean of final-layer token vectors. This is **zero-shot embedding retrieval**, not T-BERT or a fine-tuned requirements trace model. <https://github.com/microsoft/CodeBERT/blob/master/UniXcoder/unixcoder.py>

**A:** cleaned text, first 512-token window · **B:** cleaned text, all windows · **C:** original text, all windows. For B/C, a file score is the maximum of its windows. This can favor longer files, so chunk counts and a mean-window diagnostic are also exported.

**Important:** Model weights require an internet connection to Hugging Face the first time. Full checkpoint inference must be run in Colab; this environment's local verification covers the TF-IDF workflow and code preflight only.

In [5]:
import importlib.util
try:
    import torch
except ImportError:
    torch = None

if RUN_UNIXCODER == "auto":
    do_unixcoder = torch is not None and torch.cuda.is_available()
else:
    do_unixcoder = bool(RUN_UNIXCODER)
print("UniXcoder enabled:", do_unixcoder)
if not do_unixcoder:
    print("Skipping model download/inference. To run neural baselines, set RUN_UNIXCODER=True, preferably with a Colab GPU.")
if do_unixcoder:
    # Restrict to supported v4 API; Colab may otherwise carry a different major version.
    from importlib.metadata import version, PackageNotFoundError
    from packaging.version import Version
    try:
        installed_tf = Version(version("transformers"))
    except PackageNotFoundError:
        installed_tf = None
    if installed_tf is None or not (Version("4.42") <= installed_tf < Version("5")):
        import subprocess
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "transformers>=4.42,<5"])
        print("Transformers package installed/adjusted; fresh Colab runtimes work best.")

UniXcoder enabled: True
Transformers package installed/adjusted; fresh Colab runtimes work best.


In [6]:
if do_unixcoder:
    from transformers import AutoTokenizer, AutoModel, AutoConfig
    from contextlib import nullcontext
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, revision=MODEL_REVISION)
    cfg = AutoConfig.from_pretrained(MODEL_NAME, revision=MODEL_REVISION)
    cfg.is_decoder = True  # Microsoft's UniXcoder wrapper sets this before loading the checkpoint.
    cfg._attn_implementation = "eager"  # Avoid possible incompatibilities with custom 3D masks.
    model = AutoModel.from_pretrained(MODEL_NAME, revision=MODEL_REVISION, config=cfg).to(device).eval()
    special_mode_id = tokenizer.convert_tokens_to_ids("<encoder-only>")
    assert special_mode_id != tokenizer.unk_token_id, "Missing UniXcoder <encoder-only> mode token"
    assert tokenizer.cls_token_id is not None and tokenizer.sep_token_id is not None
    assert tokenizer.pad_token_id is not None
    assert 4 < MAX_TOKENS < 1024 and 0 < CHUNK_STRIDE <= MAX_TOKENS - 4
    print("Device:", device, "Hidden size:", model.config.hidden_size,
          "mode token ID:", special_mode_id, "Transformers:", __import__("transformers").__version__)
else:
    print("Model initialization skipped.")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/691 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/504M [00:00<?, ?B/s]

Device: cuda Hidden size: 768 mode token ID: 6 Transformers: 4.57.6


In [7]:
if do_unixcoder:
    @torch.no_grad()
    def token_windows(text, first_only=False):
        body = tokenizer.convert_tokens_to_ids(tokenizer.tokenize(text))
        room = MAX_TOKENS - 4
        if first_only:
            starts = [0]
        else:
            starts = list(range(0, max(len(body)-room, 0)+1, CHUNK_STRIDE))
            if starts[-1] + room < len(body):
                starts.append(len(body)-room)
        prefix = [tokenizer.cls_token_id, special_mode_id, tokenizer.sep_token_id]
        return [prefix + body[start:start+room] + [tokenizer.sep_token_id] for start in starts], len(body)

    @torch.inference_mode()
    def embed_token_windows(chunks):
        out = np.empty((len(chunks), model.config.hidden_size), dtype=np.float32)
        idx_order = np.argsort([len(c) for c in chunks], kind="stable")
        for start in range(0, len(chunks), BATCH_SIZE):
            idx = idx_order[start:start+BATCH_SIZE]
            width = max(len(chunks[i]) for i in idx)
            ids = torch.full((len(idx), width), tokenizer.pad_token_id, dtype=torch.long)
            for row, i in enumerate(idx):
                ids[row, :len(chunks[i])] = torch.tensor(chunks[i], dtype=torch.long)
            ids = ids.to(device)
            real = ids.ne(tokenizer.pad_token_id)
            # Reproduce official 3D encoder-only mask: valid queries attend to valid keys.
            attention = (real.unsqueeze(1) & real.unsqueeze(2)).long()
            context = torch.autocast(device_type="cuda", dtype=torch.float16) if USE_AMP and device.type == "cuda" else nullcontext()
            with context:
                outputs = model(input_ids=ids, attention_mask=attention)
            hidden = outputs.last_hidden_state.float()
            weights = real.unsqueeze(-1).float()
            pooled = (hidden * weights).sum(1) / weights.sum(1).clamp(min=1)
            out[idx] = torch.nn.functional.normalize(pooled, p=2, dim=1).cpu().numpy()
        return out

    def embed_documents(texts, tag, first_only=False):
        fingerprint = hashlib.sha256()
        for val in [MODEL_NAME, MODEL_REVISION, tag, str(MAX_TOKENS), str(CHUNK_STRIDE),
                    str(first_only), str(USE_AMP), "encoder3d_mean_v2.1", *texts]:
            fingerprint.update(val.encode("utf-8", errors="replace"))
            fingerprint.update(b"\0")
        hashid = fingerprint.hexdigest()[:20]
        path = Path(CACHE_DIR) / f"{tag}_{hashid}.npz"
        if path.exists():
            with np.load(path) as f:
                vec, owners, counts, lengths = (f[k] for k in ["vectors","owner","counts","lengths"])
            if len(counts) == len(texts) and vec.shape[0] == len(owners):
                print("Cache hit:", path.name)
                return vec, owners, counts, lengths
            raise RuntimeError("Invalid cache contents: " + str(path))
        windows, owners, counts, lengths = [], [], [], []
        for i, text in enumerate(texts):
            pieces, length = token_windows(text, first_only=first_only)
            windows.extend(pieces)
            owners.extend([i] * len(pieces))
            counts.append(len(pieces))
            lengths.append(length)
        print(f"Embedding {len(texts)} {tag} docs as {len(windows)} windows (max tokens {MAX_TOKENS})")
        vec = embed_token_windows(windows)
        owners = np.asarray(owners, dtype=np.int32)
        counts = np.asarray(counts, dtype=np.int32)
        lengths = np.asarray(lengths, dtype=np.int32)
        path.parent.mkdir(parents=True, exist_ok=True)
        np.savez_compressed(path, vectors=vec, owner=owners, counts=counts, lengths=lengths)
        return vec, owners, counts, lengths

    def chunk_similarity(req_vec, code_vec, owner, aggregation="max"):
        scores = req_vec @ code_vec.T
        out = np.empty((len(req_vec), len(code_ids)), dtype=np.float32)
        for j in range(len(code_ids)):
            subset = scores[:, owner == j]
            if subset.shape[1] == 0:
                raise ValueError(f"No chunks for {code_ids[j]}")
            out[:,j] = subset.max(axis=1) if aggregation == "max" else subset.mean(axis=1)
        return out
else:
    print("Neural embedding functions omitted because RUN_UNIXCODER is disabled.")

## 5. UniXcoder ablation runs
Three conditions each change one major factor. These are not controlled hyperparameter-selection experiments; comparisons on the same 105 linked requirements are **exploratory**. Cache keys include model/settings/input text to avoid stale vectors.

In [8]:
chunk_diagnostics = []
if do_unixcoder:
    variants = [
        ("UniXcoder A | cleaned, first window", req_clean, code_clean, "clean", True),
        ("UniXcoder B | cleaned, all windows", req_clean, code_clean, "clean", False),
        ("UniXcoder C | original, all windows", req_raw, code_raw, "raw", False),
    ]
    for name, req_texts, code_texts, label, first_only in variants:
        req_vec, _, req_counts, req_lengths = embed_documents(req_texts, f"req_{label}_first", first_only=True)
        code_vec, code_owner, counts, lengths = embed_documents(code_texts,
                                       f"code_{label}_{'first' if first_only else 'all'}", first_only=first_only)
        print(name, "code windows:", len(code_owner), "files with >1 window:", int((counts>1).sum()),
              "requirements exceeding first window:", int((req_lengths > MAX_TOKENS-4).sum()))
        sim = chunk_similarity(req_vec, code_vec, code_owner, aggregation="max")
        all_similarities[name] = sim
        all_results[name] = evaluate(sim)
        for j, fn in enumerate(code_ids):
            chunk_diagnostics.append({"Variant": name, "JavaFile": fn,
                                      "ChunkCount": int(counts[j]), "TokenCount": int(lengths[j])})
        if not first_only:
            diag = name + " | mean-window diagnostic"
            all_similarities[diag] = chunk_similarity(req_vec, code_vec, code_owner, aggregation="mean")
            all_results[diag] = evaluate(all_similarities[diag])

summary_all = pd.DataFrame({name: summarize(results) for name,results in all_results.items()}).T
summary_pilot = pd.DataFrame({name: summarize(results.loc[pilot_reqs]) for name,results in all_results.items()}).T
print("All 105 linked requirements:")
display(summary_all.round(4))
print("Historical 10-query subset:")
display(summary_pilot.round(4))

Embedding 131 req_clean_first docs as 131 windows (max tokens 512)


model.safetensors:   0%|          | 0.00/504M [00:00<?, ?B/s]

Embedding 226 code_clean_first docs as 226 windows (max tokens 512)
UniXcoder A | cleaned, first window code windows: 226 files with >1 window: 0 requirements exceeding first window: 0
Cache hit: req_clean_first_8194f5696bc4a3f8cb98.npz
Embedding 226 code_clean_all docs as 655 windows (max tokens 512)
UniXcoder B | cleaned, all windows code windows: 655 files with >1 window: 142 requirements exceeding first window: 0
Embedding 131 req_raw_first docs as 131 windows (max tokens 512)
Embedding 226 code_raw_all docs as 646 windows (max tokens 512)
UniXcoder C | original, all windows code windows: 646 files with >1 window: 140 requirements exceeding first window: 0
All 105 linked requirements:


,Precision@5,Recall@5,F1@5,MAP,MeanFirstHitRank
TF-IDF,0.1924,0.3893,0.2437,0.2896,14.3429
"UniXcoder A | cleaned, first window",0.0819,0.1699,0.1039,0.1420,32.9238
"UniXcoder B | cleaned, all windows",0.1829,0.3634,0.2321,0.3321,9.8286
"UniXcoder B | cleaned, all windows | mean-window diagnostic",0.1886,0.3899,0.2427,0.3038,12.8381
"UniXcoder C | original, all windows",0.1829,0.3805,0.2356,0.3067,8.8571
"UniXcoder C | original, all windows | mean-window diagnostic",0.1638,0.3563,0.2137,0.2799,15.9429


Historical 10-query subset:


,Precision@5,Recall@5,F1@5,MAP,MeanFirstHitRank
TF-IDF,0.24,0.6000,0.3178,0.3765,2.2
"UniXcoder A | cleaned, first window",0.08,0.2000,0.1051,0.1260,24.2
"UniXcoder B | cleaned, all windows",0.14,0.2583,0.1761,0.3243,12.3
"UniXcoder B | cleaned, all windows | mean-window diagnostic",0.16,0.2833,0.1983,0.3304,7.4
"UniXcoder C | original, all windows",0.20,0.4000,0.2499,0.3072,11.0
"UniXcoder C | original, all windows | mean-window diagnostic",0.16,0.3417,0.2026,0.3175,7.1


## 6. Per-query differences and transparent failure inspection
We do **not** choose a winning model on the same evaluation data and then report a confirmatory p-value. The comparisons here are descriptive/exploratory. A future knowledge-based experiment should freeze authoring evidence and settings using development/validation groups before evaluating reserved test groups.

In [9]:
if len(all_results) > 1:
    base_ap = all_results["TF-IDF"]["AP"]
    change_rows = []
    for name, metric in all_results.items():
        if name == "TF-IDF": continue
        delta = metric["AP"] - base_ap
        change_rows.append({"Model": name,
                            "MAP_delta_vs_TFIDF": delta.mean(),
                            "QueriesImproved": int((delta > 1e-10).sum()),
                            "QueriesWorsened": int((delta < -1e-10).sum()),
                            "QueriesTied": int((delta.abs() <= 1e-10).sum())})
    display(pd.DataFrame(change_rows).round(4))
    # Fixed comparison to prevent post-hoc selection of the 'best' variant.
    focus_name = "UniXcoder C | original, all windows"
    delta = all_results[focus_name]["AP"] - base_ap
    changes = pd.DataFrame({"TFIDF_AP": base_ap, "UniXcoderC_AP": all_results[focus_name]["AP"],
                            "Delta": delta}).sort_values("Delta")
    print("Five cases where TF-IDF did better than UniXcoder C:")
    display(changes.head(5).round(4))
    print("Five cases where UniXcoder C did better than TF-IDF:")
    display(changes.tail(5).round(4))
    inspection_id = changes.index[0]
    print("Inspecting requirement", inspection_id, "\n", requirements[inspection_id][:1500])
    print("Recorded Java links:", sorted(ground_truth[inspection_id]))
    display(pd.concat({"TF-IDF": top_matches(all_similarities["TF-IDF"],inspection_id),
                       "UniXcoder C": top_matches(all_similarities[focus_name],inspection_id)},axis=1))
else:
    print("TF-IDF-only run. Re-enable UniXcoder to obtain model comparisons.")

print("Worked example", PILOT_REQ, "(recorded links):")
for name, scores in all_similarities.items():
    display(name, top_matches(scores, PILOT_REQ, n=5))

,Model,MAP_delta_vs_TFIDF,QueriesImproved,QueriesWorsened,QueriesTied
0,"UniXcoder A | cleaned, first window",-0.1476,16,86,3
1,"UniXcoder B | cleaned, all windows",0.0425,58,45,2
2,"UniXcoder B | cleaned, all windows | mean-wind...",0.0142,50,51,4
3,"UniXcoder C | original, all windows",0.0171,57,43,5
4,"UniXcoder C | original, all windows | mean-win...",-0.0097,48,52,5


Five cases where TF-IDF did better than UniXcoder C:


,TFIDF_AP,UniXcoderC_AP,Delta
Requirement,,,
UC34E3.txt,1.0000,0.0139,-0.9861
UC38S1.txt,0.7500,0.2576,-0.4924
UC34E2.txt,0.5000,0.0145,-0.4855
UC23S1.txt,0.5167,0.0451,-0.4716
UC18E1.txt,0.5000,0.0435,-0.4565


Five cases where UniXcoder C did better than TF-IDF:


,TFIDF_AP,UniXcoderC_AP,Delta
Requirement,,,
UC3S2.txt,0.2361,0.7333,0.4972
UC18S2.txt,0.3250,0.8333,0.5083
UC6S1.txt,0.0522,0.5783,0.5260
UC18S1.txt,0.2262,0.8333,0.6071
UC6S2.txt,0.0278,1.0000,0.9722


Inspecting requirement UC34E3.txt 
 The patient, UAP, or personal representative enters a glucose level outside the range 0-250. He/she is notified of an error and is prompted to try again.

Recorded Java links: ['RemoteMonitoringDataBeanValidator.java']


... (10 outputs left out)

'UniXcoder C | original, all windows'

,Rank,JavaFile,Similarity,RecordedLink
0,1,AuthDAO.java,0.429297,False
1,2,ResetPasswordAction.java,0.395855,True
2,3,LabProcUAPAction.java,0.272733,False
3,4,EditOfficeVisitAction.java,0.263881,False
4,5,TransactionType.java,0.261348,False


'UniXcoder C | original, all windows | mean-window diagnostic'

,Rank,JavaFile,Similarity,RecordedLink
0,1,ResetPasswordAction.java,0.272846,True
1,2,AuthDAO.java,0.265168,False
2,3,TransactionType.java,0.219988,False
3,4,ReferralManagementAction.java,0.214651,False
4,5,UpdateCPTCodeListAction.java,0.210827,False


## 7. Export artifacts and run manifest
Download `DoCIT_iTrust_Task1_results.zip` and retain it alongside the executed Colab notebook. It contains all ranked candidates (not just top-5), summary metrics, individual-query metrics, and a reproducibility manifest. It does **not** contain or distribute source-code files from the dataset.

In [10]:
import datetime, shutil

summary_all.to_csv(output_dir / "metrics_all_105.csv", float_format="%.8f")
summary_pilot.to_csv(output_dir / "metrics_original_10.csv", float_format="%.8f")
pd.concat(all_results, names=["Model", "Requirement"]).to_csv(output_dir / "per_requirement_metrics.csv", float_format="%.8f")

with (output_dir / "ranked_candidates.csv").open("w", encoding="utf-8") as f:
    f.write("Model,Requirement,Rank,JavaFile,Similarity,RecordedLink\n")
    import csv
    writer = csv.writer(f)
    for name, sim in all_similarities.items():
        for rid in eval_reqs:
            order = np.argsort(-sim[req_pos[rid]], kind="stable")
            for rank, j in enumerate(order, 1):
                writer.writerow([name, rid, rank, code_ids[j], float(sim[req_pos[rid],j]),
                                 int(code_ids[j] in ground_truth[rid])])
if chunk_diagnostics:
    pd.DataFrame(chunk_diagnostics).to_csv(output_dir / "code_chunk_diagnostics.csv", index=False)

manifest = {
    "project": "Verified Domain Knowledge for Requirements-to-Code Traceability",
    "scope": "Exploratory baseline and ablation; no knowledge-assisted retrieval implemented",
    "timestamp_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "zip_filename": zip_path.name, "archive_sha256": archive_sha256,
    "requirements": len(req_ids), "java_candidates": len(code_ids),
    "requirements_with_reference_links": len(eval_reqs),
    "recorded_reference_links": sum(map(len,ground_truth.values())),
    "queries_without_reference_links": len(req_ids)-len(eval_reqs),
    "pilot_reqs": pilot_reqs, "K": K,
    "tfidf": {"stop_words": "english", "ngram_range": [1,2], "sublinear_tf": True,
              "vocab_size": len(vectorizer.vocabulary_), "fit_scope": "all 131 requirements plus all 226 Java files"},
    "unixcoder_run": bool(do_unixcoder), "model_name": MODEL_NAME if do_unixcoder else None,
    "model_revision_request": MODEL_REVISION if do_unixcoder else None,
    "model_resolved_commit": getattr(model.config, "_commit_hash", None) if do_unixcoder else None,
    "max_tokens": MAX_TOKENS if do_unixcoder else None,
    "stride": CHUNK_STRIDE if do_unixcoder else None,
    "batch_size": BATCH_SIZE if do_unixcoder else None,
    "mixed_precision": USE_AMP if do_unixcoder else None,
    "neural_pooling": "masked mean of final-layer token vectors" if do_unixcoder else None,
    "aggregation": "max chunk; mean chunk diagnostic" if do_unixcoder else None,
    "models_evaluated": list(all_results),
    "packages": {"python": platform.python_version(),"numpy": np.__version__,"pandas": pd.__version__,
                 "scikit_learn": sklearn.__version__,"scipy": scipy.__version__,
                 "torch": getattr(torch,"__version__",None),
                 "transformers": __import__("transformers").__version__ if do_unixcoder else None},
    "limitations": ["Evaluation uses the 105 requirements that have recorded links; link matrix may be incomplete.",
                    "Baseline TF-IDF is transductive (fitted all requirements and code) and not a held-out knowledge experiment.",
                    "All UniXcoder variant comparisons are exploratory; no confirmatory test/p-value is asserted.",
                    "Max-of-chunks may reward larger Java files; mean-window diagnostic is provided."]
}
(output_dir/"run_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
archive_path = shutil.make_archive(str(work/"DoCIT_iTrust_Task1_results"),"zip",root_dir=output_dir)
print("Outputs:", output_dir)
print("Downloadable ZIP:", archive_path, "size:", round(Path(archive_path).stat().st_size/1048576,2), "MB")
print("Models scored:", len(all_results), "; full ranked pairs:", len(eval_reqs)*len(code_ids)*len(all_results))

Outputs: /content/docit_task1/results
Downloadable ZIP: /content/docit_task1/DoCIT_iTrust_Task1_results.zip size: 2.13 MB
Models scored: 6 ; full ranked pairs: 142380


## 8. What is completed versus what remains in the original DoCIT proposal
**Completed by this notebook (after a run):** original iTrust data validation, TF-IDF baseline and optionally UniXcoder A/B/C, reproducible ranks and metrics, inspected failure cases, and metadata.

**Not implemented or established by this notebook:** LLM-assisted glossary/relationship extraction; source-evidence verification and human acceptance; the three controlled conditions (**context disabled**, **verified glossary only**, **glossary + verified code relationships**); reserved validation/test comparisons; or LLM-versus-manual knowledge-authoring effort. Those are separate research activities and must never be described as demonstrated improvements.

**Recommended next experimental phase:** (1) define use-case-group partitions; (2) collect candidate knowledge only from development requirements and permitted Java code; (3) independently verify every accepted concept/relationship; (4) freeze the knowledge and scoring weights with validation data; (5) evaluate the reserved test set once, comparing the three knowledge conditions against the same baseline and candidates.